In [ ]:
import requests

url = "https://disseminate.stats.swiss/rest/dataflow/CH1.COU/DF_COU_HEALTH_COSTS/1.0.0?references=all"

response = requests.get(url)
response.raise_for_status()

with open("dataflow.xml", "wb") as f:
    f.write(response.content)

from lxml import etree

tree = etree.fromstring(response.content)

pretty_xml = etree.tostring(tree, pretty_print=True, encoding="unicode")

with open("dataflow_pretty.xml", "w", encoding="utf-8") as f:
    f.write(pretty_xml)

print(pretty_xml[:3000])

In [ ]:
from lxml import etree
import pandas as pd

ns = {
    "str": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure",
    "com": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common",
}

tree = etree.parse("dataflow.xml")

rows = []

for dim in tree.xpath("//str:Dimension", namespaces=ns):
    dim_id = dim.get("id")

    concept_ref = dim.find(".//str:ConceptIdentity/Ref", namespaces=ns)
    concept_id = concept_ref.get("id") if concept_ref is not None else None
    concept_scheme = concept_ref.get("maintainableParentID") if concept_ref is not None else None

    codelist_ref = dim.find(".//str:LocalRepresentation/str:Enumeration/Ref", namespaces=ns)
    codelist_id = codelist_ref.get("id") if codelist_ref is not None else None

    rows.append({
        "dimension_id": dim_id,
        "concept_id": concept_id,
        "concept_scheme": concept_scheme,
        "codelist_id": codelist_id,
    })

df_dims = pd.DataFrame(rows)
df_dims

In [ ]:
concept_rows = []

for concept in tree.xpath("//str:Concept", namespaces=ns):
    concept_id = concept.get("id")
    names = concept.xpath("./com:Name", namespaces=ns)
    name = names[0].text if names else None

    concept_rows.append({
        "concept_id": concept_id,
        "description": name,
    })

df_concepts = pd.DataFrame(concept_rows)
df_concepts

In [ ]:
df_dict = df_dims.merge(
    df_concepts[["concept_id", "description"]],
    on="concept_id",
    how="left"
)

df_dict

In [ ]:
code_rows = []

for cl in tree.xpath("//str:Codelist", namespaces=ns):
    cl_id = cl.get("id")

    for code in cl.xpath("./str:Code", namespaces=ns):
        code_id = code.get("id")
        names = code.xpath("./com:Name", namespaces=ns)
        name = names[0].text if names else None

        code_rows.append({
            "codelist_id": cl_id,
            "code": code_id,
            "description": name,
        })

df_codes = pd.DataFrame(code_rows)
df_codes

In [ ]:
df_codes.loc[df_codes['codelist_id'] == "CL_OBS_STATUS"]